In [ ]:
from langchain.document_loaders import PyPDFLoader, OnlinePDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
import os

In [3]:
loader = PyPDFLoader("/kaggle/input/attention/transformer.pdf")

In [4]:
data = loader.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap = 0)
docs = text_splitter.split_documents(data)

In [6]:
len(docs)

91

In [7]:
docs[0]

Document(page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.comNoam Shazeer∗\nGoogle Brain\nnoam@google.comNiki Parmar∗\nGoogle Research\nnikip@google.comJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.comAidan N. Gomez∗ †\nUniversity of Toronto', metadata={'source': '/kaggle/input/attention/transformer.pdf', 'page': 0})

In [8]:
HUGGINGFACEHUB_API_TOKEN = os.environ.get['HUGGINGFACEHUB_API_TOKEN']
PINECONE_API_KEY = os.environ.get['PINECONE_API_KEY']
PINECONE_API_ENV = os.environ['PINECONE_API_ENV']

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs = {"device" : "cuda"})

.gitattributes:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

train_script.py:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

In [10]:
import pinecone
pc = pinecone.Pinecone(api_key = os.environ.get('PINECONE_API_KEY'), api_env = os.environ.get('PINECONE_API_ENV'),)
index_name ='llama2'

In [11]:
from langchain.vectorstores import Pinecone

In [12]:
texts = [t.page_content for t in docs]

In [13]:
# docsearch = Pinecone.from_texts(texts, embeddings,index_name=index_name)
docsearch = Pinecone.from_existing_index(index_name,embeddings)

In [14]:
docsearch.similarity_search("transformer")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Document(page_content='Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N= 6 identical layers. Each layer has two\nsub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-'),
 Document(page_content='the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Transformer models and\nhas been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head\nattention and the parameter-free position representation and became the other person involved in nearly every\ndetail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and'),
 Documen

In [ ]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python numpy --force-reinstall --upgrade --no-cache-dir --verbose

In [16]:
from langchain.llms import LlamaCpp
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from huggingface_hub import hf_hub_download
from langchain.chains.question_answering import load_qa_chain

In [17]:
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

In [18]:
model_name = 'TheBloke/Llama-2-13B-chat-GGUF'
# model_basename = "llama-2-13b-chat.ggmlv3.q5_1.bin"
model_basename = "llama-2-13b-chat.Q8_0.gguf"

In [19]:
model_path = hf_hub_download(repo_id = model_name,filename = model_basename)

llama-2-13b-chat.Q8_0.gguf:   0%|          | 0.00/13.8G [00:00<?, ?B/s]

In [20]:
model_path

'/root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q8_0.gguf'

In [21]:
n_gpu_layers = 40  # Change this value based on your model and your GPU VRAM pool.
n_batch = 256  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.

# Loading model,
llm = LlamaCpp(
    model_path=model_path,
    max_tokens=512,
    n_gpu_layers=n_gpu_layers,
    n_batch=n_batch,
    callback_manager=callback_manager,
    n_ctx=20480,
    verbose=True,
)

llama_model_loader: loaded meta data with 19 key-value pairs and 363 tensors from /root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q8_0.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 5120
llama_model_loader: - kv   4:                          llama.block_count u32              = 40
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 13824
llama_model_loader: - kv   6:                 llama.rope.dimension_co

In [22]:
chain = load_qa_chain(llm , chain_type="stuff")

In [23]:
query="How transformers work"
docs=docsearch.similarity_search(query)
docs

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Document(page_content='Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N= 6 identical layers. Each layer has two\nsub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-'),
 Document(page_content='the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Transformer models and\nhas been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head\nattention and the parameter-free position representation and became the other person involved in nearly every\ndetail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and'),
 Documen

In [24]:

chain.run(input_documents = docs, question = query)

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `run` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


 Transformers work using self-attention and point-wise, fully connected layers, with a stacked architecture that includes an encoder and decoder. The encoder is composed of six identical layers, each with two sub-layers: a multi-head self-attention mechanism and a simple position-wise feed-forward network. The decoder is also composed of six identical layers, but with an additional attention layer to allow it to attend to the output of the encoder. The model uses scaled dot-product attention and multi-head attention to learn dependencies between distant positions, and has a parameter-free position representation to reduce the number of operations required.


llama_print_timings:        load time =  186605.43 ms
llama_print_timings:      sample time =     287.50 ms /   137 runs   (    2.10 ms per token,   476.53 tokens per second)
llama_print_timings: prompt eval time =  453254.05 ms /   594 tokens (  763.05 ms per token,     1.31 tokens per second)
llama_print_timings:        eval time = 6120168.23 ms /   136 runs   (45001.24 ms per token,     0.02 tokens per second)
llama_print_timings:       total time = 6574760.08 ms /   730 tokens


' Transformers work using self-attention and point-wise, fully connected layers, with a stacked architecture that includes an encoder and decoder. The encoder is composed of six identical layers, each with two sub-layers: a multi-head self-attention mechanism and a simple position-wise feed-forward network. The decoder is also composed of six identical layers, but with an additional attention layer to allow it to attend to the output of the encoder. The model uses scaled dot-product attention and multi-head attention to learn dependencies between distant positions, and has a parameter-free position representation to reduce the number of operations required.'